# Inspect InaTRC labels — normalized YOLO cxcywh

Run the InaTRC preparation command in the README before opening this notebook. It draws labels from the frozen training split in original image coordinates.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt

from detectit.data.boxes import normalized_cxcywh_to_xyxy
from detectit.data.dataset import check_records, load_manifest, read_labels
from detectit.models.yolo import Detection
from detectit.utils.drawing import draw_detections

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
manifest = load_manifest(root / "data/inatrc/splits.json")
records = check_records(manifest, "train")[:4]
names = manifest["names"]


In [ ]:
figure, axes = plt.subplots(2, 2, figsize=(14, 9), layout="constrained")
for axis, record in zip(axes.flat, records, strict=True):
    image = cv2.imread(str(manifest["root"] / record["image"]))
    height, width = image.shape[:2]
    labels = read_labels(manifest["root"] / record["label"], len(names))
    boxes = [
        Detection(class_id, names[class_id], 1.0, normalized_cxcywh_to_xyxy(box, width, height))
        for class_id, box in labels
    ]
    axis.imshow(cv2.cvtColor(draw_detections(image, boxes, show_confidence=False), cv2.COLOR_BGR2RGB))
    axis.set_title(Path(record["image"]).name)
    axis.axis("off")
